In [1]:
dbutils.widgets.text("init_load_flag", "True")

Box(children=(Label(value='init_load_flag'), Text(value='True')))

In [2]:
init_load_flag = bool(dbutils.widgets.get("init_load_flag"))
init_load_flag

True

In [3]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Data Reading from Source

In [4]:
df = spark.sql(
    """
    SELECT * FROM learn_e2e_1.silver.customers
    """
)
df

,customer_id,email,city,state,domains,full_name
0,C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney
1,C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers
2,C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes
3,C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott
4,C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez
5,C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy
6,C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle
7,C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos
8,C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green
9,C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers


Removing Duplicates

In [5]:
df = df.dropDuplicates(subset=["customer_id"])
df

,customer_id,email,city,state,domains,full_name
0,C01220,matthew01@yahoo.com,New Andrewhaven,WA,yahoo.com,Thomas Fitzgerald
1,C01579,nathancastro@gmail.com,South Amanda,FL,gmail.com,Brittany Schmidt
2,C01155,cynthia51@lewis-dixon.com,East Theresa,FL,lewis-dixon.com,Latasha Phelps
3,C01943,rodriguezzachary@hotmail.com,East Timothy,PA,hotmail.com,David Cooper
4,C01554,sthompson@harding.com,New Leefurt,ME,harding.com,Gavin Lindsey
5,C00767,rsmith@pruitt-hodges.net,Heatherton,IL,pruitt-hodges.net,Pam Watts
6,C01097,joel22@stone-holmes.com,Davidhaven,LA,stone-holmes.com,James Martinez
7,C01674,crystalraymond@keller.com,Hoborough,ME,keller.com,Nicole Johnston
8,C00215,petersonthomas@yahoo.com,South Andrea,OK,yahoo.com,Danielle Huerta
9,C00587,phillipsstephanie@gmail.com,New Ronaldmouth,OR,gmail.com,Michael Bailey


** Surrogate Key - All The Values **

In [6]:
df = df.withColumn("DimCustomerKey", monotonically_increasing_id()+lit(1))
df

,customer_id,email,city,state,domains,full_name,DimCustomerKey
0,C01220,matthew01@yahoo.com,New Andrewhaven,WA,yahoo.com,Thomas Fitzgerald,1
1,C01579,nathancastro@gmail.com,South Amanda,FL,gmail.com,Brittany Schmidt,2
2,C01155,cynthia51@lewis-dixon.com,East Theresa,FL,lewis-dixon.com,Latasha Phelps,3
3,C01943,rodriguezzachary@hotmail.com,East Timothy,PA,hotmail.com,David Cooper,4
4,C01554,sthompson@harding.com,New Leefurt,ME,harding.com,Gavin Lindsey,5
5,C00767,rsmith@pruitt-hodges.net,Heatherton,IL,pruitt-hodges.net,Pam Watts,6
6,C01097,joel22@stone-holmes.com,Davidhaven,LA,stone-holmes.com,James Martinez,7
7,C01674,crystalraymond@keller.com,Hoborough,ME,keller.com,Nicole Johnston,8
8,C00215,petersonthomas@yahoo.com,South Andrea,OK,yahoo.com,Danielle Huerta,9
9,C00587,phillipsstephanie@gmail.com,New Ronaldmouth,OR,gmail.com,Michael Bailey,10


## Dividing New vs Old Records

In [7]:
if init_load_flag == False:
    df_old = spark.sql(
        """
            SELECT DimCustomerKey, customer_id, create_date, update_date
            FROM learn_e2e_1.gold.customers
        """)

else:
    df_old = spark.sql(
        """
            SELECT 0 DimCustomerKey, 0 customer_id, 0 create_date, 0 update_date 
            FROM learn_e2e_1.silver.customers
            WHERE 1=0
        """)
    
df_old

,DimCustomerKey,customer_id,create_date,update_date


#### Renaming Columns of df_old

In [8]:
df_old = df_old.withColumnRenamed("DimCustomerKey", "old_DimCustomerKey") \
    .withColumnRenamed("customer_id", "old_customer_id") \
    .withColumnRenamed("create_date", "old_create_date") \
    .withColumnRenamed("update_date", "old_update_date")
df_old

,old_DimCustomerKey,old_customer_id,old_create_date,old_update_date


#### Applying Join with Old Records

In [9]:
df_join = df.join(df_old, df['customer_id'] == df_old['old_customer_id'], "left")
df_join

,customer_id,email,city,state,domains,full_name,DimCustomerKey,old_DimCustomerKey,old_customer_id,old_create_date,old_update_date
0,C01220,matthew01@yahoo.com,New Andrewhaven,WA,yahoo.com,Thomas Fitzgerald,1,NaN,NaN,NaN,NaN
1,C01579,nathancastro@gmail.com,South Amanda,FL,gmail.com,Brittany Schmidt,2,NaN,NaN,NaN,NaN
2,C01155,cynthia51@lewis-dixon.com,East Theresa,FL,lewis-dixon.com,Latasha Phelps,3,NaN,NaN,NaN,NaN
3,C01943,rodriguezzachary@hotmail.com,East Timothy,PA,hotmail.com,David Cooper,4,NaN,NaN,NaN,NaN
4,C01554,sthompson@harding.com,New Leefurt,ME,harding.com,Gavin Lindsey,5,NaN,NaN,NaN,NaN
5,C00767,rsmith@pruitt-hodges.net,Heatherton,IL,pruitt-hodges.net,Pam Watts,6,NaN,NaN,NaN,NaN
6,C01097,joel22@stone-holmes.com,Davidhaven,LA,stone-holmes.com,James Martinez,7,NaN,NaN,NaN,NaN
7,C01674,crystalraymond@keller.com,Hoborough,ME,keller.com,Nicole Johnston,8,NaN,NaN,NaN,NaN
8,C00215,petersonthomas@yahoo.com,South Andrea,OK,yahoo.com,Danielle Huerta,9,NaN,NaN,NaN,NaN
9,C00587,phillipsstephanie@gmail.com,New Ronaldmouth,OR,gmail.com,Michael Bailey,10,NaN,NaN,NaN,NaN


#### Separating New vs Old Records

In [10]:
df_new = df_join.filter(df_join['old_DimCustomerKey'].isNull())
df_new

,customer_id,email,city,state,domains,full_name,DimCustomerKey,old_DimCustomerKey,old_customer_id,old_create_date,old_update_date
0,C01220,matthew01@yahoo.com,New Andrewhaven,WA,yahoo.com,Thomas Fitzgerald,1,NaN,NaN,NaN,NaN
1,C01579,nathancastro@gmail.com,South Amanda,FL,gmail.com,Brittany Schmidt,2,NaN,NaN,NaN,NaN
2,C01155,cynthia51@lewis-dixon.com,East Theresa,FL,lewis-dixon.com,Latasha Phelps,3,NaN,NaN,NaN,NaN
3,C01943,rodriguezzachary@hotmail.com,East Timothy,PA,hotmail.com,David Cooper,4,NaN,NaN,NaN,NaN
4,C01554,sthompson@harding.com,New Leefurt,ME,harding.com,Gavin Lindsey,5,NaN,NaN,NaN,NaN
5,C00767,rsmith@pruitt-hodges.net,Heatherton,IL,pruitt-hodges.net,Pam Watts,6,NaN,NaN,NaN,NaN
6,C01097,joel22@stone-holmes.com,Davidhaven,LA,stone-holmes.com,James Martinez,7,NaN,NaN,NaN,NaN
7,C01674,crystalraymond@keller.com,Hoborough,ME,keller.com,Nicole Johnston,8,NaN,NaN,NaN,NaN
8,C00215,petersonthomas@yahoo.com,South Andrea,OK,yahoo.com,Danielle Huerta,9,NaN,NaN,NaN,NaN
9,C00587,phillipsstephanie@gmail.com,New Ronaldmouth,OR,gmail.com,Michael Bailey,10,NaN,NaN,NaN,NaN


In [11]:
df_update = df_join.filter(df_join['old_DimCustomerKey'].isNotNull())
df_update

,customer_id,email,city,state,domains,full_name,DimCustomerKey,old_DimCustomerKey,old_customer_id,old_create_date,old_update_date


#### Preparing df_old

In [12]:
# Droppping all the columns which are not required
df_update = df_update.drop('old_DimCustomerKey', 'old_customer_id', 'old_update_date')

# Renaming "old_create_date" to "create_date"
df_update = df_update.withColumnRenamed("old_create_date", "create_date")
df_update = df_update.withColumn("create_date", to_timestamp(col("create_date")))

# Recreating "update_date" column
df_update = df_update.withColumn("update_date", current_timestamp())
df_update


,customer_id,email,city,state,domains,full_name,DimCustomerKey,create_date,update_date


#### Preparing df_new

In [13]:
# Dropping all the columns which are not required
df_new = df_new.drop("old_DimCustomerKey", "old_customer_id", "old_create_date", "old_update_date")

#Recreating "create_date" and "update_date" columns
df_new = df_new.withColumn("create_date", current_timestamp()) \
    .withColumn("update_date", current_timestamp())
    
df_new

,customer_id,email,city,state,domains,full_name,DimCustomerKey,create_date,update_date
0,C01220,matthew01@yahoo.com,New Andrewhaven,WA,yahoo.com,Thomas Fitzgerald,1,2025-07-17 11:35:40.552605,2025-07-17 11:35:40.552605
1,C01579,nathancastro@gmail.com,South Amanda,FL,gmail.com,Brittany Schmidt,2,2025-07-17 11:35:40.552605,2025-07-17 11:35:40.552605
2,C01155,cynthia51@lewis-dixon.com,East Theresa,FL,lewis-dixon.com,Latasha Phelps,3,2025-07-17 11:35:40.552605,2025-07-17 11:35:40.552605
3,C01943,rodriguezzachary@hotmail.com,East Timothy,PA,hotmail.com,David Cooper,4,2025-07-17 11:35:40.552605,2025-07-17 11:35:40.552605
4,C01554,sthompson@harding.com,New Leefurt,ME,harding.com,Gavin Lindsey,5,2025-07-17 11:35:40.552605,2025-07-17 11:35:40.552605
5,C00767,rsmith@pruitt-hodges.net,Heatherton,IL,pruitt-hodges.net,Pam Watts,6,2025-07-17 11:35:40.552605,2025-07-17 11:35:40.552605
6,C01097,joel22@stone-holmes.com,Davidhaven,LA,stone-holmes.com,James Martinez,7,2025-07-17 11:35:40.552605,2025-07-17 11:35:40.552605
7,C01674,crystalraymond@keller.com,Hoborough,ME,keller.com,Nicole Johnston,8,2025-07-17 11:35:40.552605,2025-07-17 11:35:40.552605
8,C00215,petersonthomas@yahoo.com,South Andrea,OK,yahoo.com,Danielle Huerta,9,2025-07-17 11:35:40.552605,2025-07-17 11:35:40.552605
9,C00587,phillipsstephanie@gmail.com,New Ronaldmouth,OR,gmail.com,Michael Bailey,10,2025-07-17 11:35:40.552605,2025-07-17 11:35:40.552605


#### Surrogate Key - From 1

In [14]:
df_new = df_new.withColumn("DimCustomerKey", monotonically_increasing_id() + lit(1))
df_new

,customer_id,email,city,state,domains,full_name,DimCustomerKey,create_date,update_date
0,C01220,matthew01@yahoo.com,New Andrewhaven,WA,yahoo.com,Thomas Fitzgerald,1,2025-07-17 11:35:42.889054,2025-07-17 11:35:42.889054
1,C01579,nathancastro@gmail.com,South Amanda,FL,gmail.com,Brittany Schmidt,2,2025-07-17 11:35:42.889054,2025-07-17 11:35:42.889054
2,C01155,cynthia51@lewis-dixon.com,East Theresa,FL,lewis-dixon.com,Latasha Phelps,3,2025-07-17 11:35:42.889054,2025-07-17 11:35:42.889054
3,C01943,rodriguezzachary@hotmail.com,East Timothy,PA,hotmail.com,David Cooper,4,2025-07-17 11:35:42.889054,2025-07-17 11:35:42.889054
4,C01554,sthompson@harding.com,New Leefurt,ME,harding.com,Gavin Lindsey,5,2025-07-17 11:35:42.889054,2025-07-17 11:35:42.889054
5,C00767,rsmith@pruitt-hodges.net,Heatherton,IL,pruitt-hodges.net,Pam Watts,6,2025-07-17 11:35:42.889054,2025-07-17 11:35:42.889054
6,C01097,joel22@stone-holmes.com,Davidhaven,LA,stone-holmes.com,James Martinez,7,2025-07-17 11:35:42.889054,2025-07-17 11:35:42.889054
7,C01674,crystalraymond@keller.com,Hoborough,ME,keller.com,Nicole Johnston,8,2025-07-17 11:35:42.889054,2025-07-17 11:35:42.889054
8,C00215,petersonthomas@yahoo.com,South Andrea,OK,yahoo.com,Danielle Huerta,9,2025-07-17 11:35:42.889054,2025-07-17 11:35:42.889054
9,C00587,phillipsstephanie@gmail.com,New Ronaldmouth,OR,gmail.com,Michael Bailey,10,2025-07-17 11:35:42.889054,2025-07-17 11:35:42.889054


#### Adding Max Surrogate Key

In [15]:
if init_load_flag == True:
    max_surrogate_key = 0
else:
    df_maxsur = spark.sql(
        """
            SELECT max(DimCustomerKey) AS max_surrogate_key
            FROM learn_e2e_1.gold.customers
        """
    )
    
    #### Converting df_maxsur to max_surrogate_key variable
    max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']
    
max_surrogate_key

df_new

,customer_id,email,city,state,domains,full_name,DimCustomerKey,create_date,update_date
0,C01220,matthew01@yahoo.com,New Andrewhaven,WA,yahoo.com,Thomas Fitzgerald,1,2025-07-17 11:35:44.827958,2025-07-17 11:35:44.827958
1,C01579,nathancastro@gmail.com,South Amanda,FL,gmail.com,Brittany Schmidt,2,2025-07-17 11:35:44.827958,2025-07-17 11:35:44.827958
2,C01155,cynthia51@lewis-dixon.com,East Theresa,FL,lewis-dixon.com,Latasha Phelps,3,2025-07-17 11:35:44.827958,2025-07-17 11:35:44.827958
3,C01943,rodriguezzachary@hotmail.com,East Timothy,PA,hotmail.com,David Cooper,4,2025-07-17 11:35:44.827958,2025-07-17 11:35:44.827958
4,C01554,sthompson@harding.com,New Leefurt,ME,harding.com,Gavin Lindsey,5,2025-07-17 11:35:44.827958,2025-07-17 11:35:44.827958
5,C00767,rsmith@pruitt-hodges.net,Heatherton,IL,pruitt-hodges.net,Pam Watts,6,2025-07-17 11:35:44.827958,2025-07-17 11:35:44.827958
6,C01097,joel22@stone-holmes.com,Davidhaven,LA,stone-holmes.com,James Martinez,7,2025-07-17 11:35:44.827958,2025-07-17 11:35:44.827958
7,C01674,crystalraymond@keller.com,Hoborough,ME,keller.com,Nicole Johnston,8,2025-07-17 11:35:44.827958,2025-07-17 11:35:44.827958
8,C00215,petersonthomas@yahoo.com,South Andrea,OK,yahoo.com,Danielle Huerta,9,2025-07-17 11:35:44.827958,2025-07-17 11:35:44.827958
9,C00587,phillipsstephanie@gmail.com,New Ronaldmouth,OR,gmail.com,Michael Bailey,10,2025-07-17 11:35:44.827958,2025-07-17 11:35:44.827958


In [16]:
df_new = df_new.withColumn("DimCustomerKey",lit(max_surrogate_key) + col("DimCustomerKey"))

#### Union of df_old and df_new

In [17]:
df_final = df_new.unionByName(df_update)
df_final

,customer_id,email,city,state,domains,full_name,DimCustomerKey,create_date,update_date
0,C01220,matthew01@yahoo.com,New Andrewhaven,WA,yahoo.com,Thomas Fitzgerald,1,2025-07-17 11:35:46.964582,2025-07-17 11:35:46.964582
1,C01579,nathancastro@gmail.com,South Amanda,FL,gmail.com,Brittany Schmidt,2,2025-07-17 11:35:46.964582,2025-07-17 11:35:46.964582
2,C01155,cynthia51@lewis-dixon.com,East Theresa,FL,lewis-dixon.com,Latasha Phelps,3,2025-07-17 11:35:46.964582,2025-07-17 11:35:46.964582
3,C01943,rodriguezzachary@hotmail.com,East Timothy,PA,hotmail.com,David Cooper,4,2025-07-17 11:35:46.964582,2025-07-17 11:35:46.964582
4,C01554,sthompson@harding.com,New Leefurt,ME,harding.com,Gavin Lindsey,5,2025-07-17 11:35:46.964582,2025-07-17 11:35:46.964582
5,C00767,rsmith@pruitt-hodges.net,Heatherton,IL,pruitt-hodges.net,Pam Watts,6,2025-07-17 11:35:46.964582,2025-07-17 11:35:46.964582
6,C01097,joel22@stone-holmes.com,Davidhaven,LA,stone-holmes.com,James Martinez,7,2025-07-17 11:35:46.964582,2025-07-17 11:35:46.964582
7,C01674,crystalraymond@keller.com,Hoborough,ME,keller.com,Nicole Johnston,8,2025-07-17 11:35:46.964582,2025-07-17 11:35:46.964582
8,C00215,petersonthomas@yahoo.com,South Andrea,OK,yahoo.com,Danielle Huerta,9,2025-07-17 11:35:46.964582,2025-07-17 11:35:46.964582
9,C00587,phillipsstephanie@gmail.com,New Ronaldmouth,OR,gmail.com,Michael Bailey,10,2025-07-17 11:35:46.964582,2025-07-17 11:35:46.964582


#### SCD Type - 1

In [18]:
from delta.tables import DeltaTable

In [20]:
if spark.catalog.tableExists("learn_e2e_1.gold.customers"):
    dlt_object = DeltaTable.forPath(spark, "s3://learn-databricks-project-e2e-1-gold/dimCustomers") 
    
    dlt_object.alias("tgt").merge(df_final.alias("src"), "tgt.DimCustomerKey = src.DimCustomerKey") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

else:
    df_final.write.mode("overwrite") \
        .format("delta") \
        .option("path", "s3://learn-databricks-project-e2e-1-gold/dimCustomers") \
        .saveAsTable("learn_e2e_1.gold.customers")
    
